<div class='bar_title'></div>

*Simulation for Decision Making (S4DM)*

# Exercise 1: Coffee Shop Simulation

Summer Semester 26


Gunther Gust & Govind Rao <br>
Chair for Enterprise AI <br>
Data Driven Decisions Group <br>
Center for Artificial Intelligence and Data Science (CAIDAS)

<img src="images/d3.png" style="width:20%; float:left;" />

<img src="images/CAIDASlogo.png" style="width:20%; float:left;" />

## Learning Goals

After completing this exercise, you should be able to:

- **Build a simulation model from scratch** following the modeling conventions (Resource class, Entity class, Generator function)
- **Use the three types of yield** in SimPy: `yield env.timeout(...)`, `yield env.process(...)`, `yield resource.request()`
- **Apply the 5-step simulation pattern** to set up and run a simulation
- **Collect data** using the `EventLogger` class with `**kwargs`
- **Analyze simulation output** using pandas and lets-plot
- **Conduct what-if analysis** by changing parameters and comparing results

## Scenario

A coffee shop has a limited number of **baristas** who prepare drinks for customers. Customers arrive throughout the day, wait in line if all baristas are busy, get their drink prepared, and leave.

**Parameters:**

| Parameter | Value | Description |
|---|---|---|
| `NUM_BARISTAS` | 2 | Number of baristas working |
| `PREP_TIME` | 5 | Minutes to prepare a drink |
| `T_INTER` | 3 | Average minutes between customer arrivals |
| `SIM_TIME` | 60 | Simulation duration in minutes |
| `RANDOM_SEED` | 42 | For reproducibility |

Your goal is to build this simulation step by step, following the **modeling conventions** from the lecture.

In [1]:
import random
import simpy
import pandas as pd
from lets_plot import *
LetsPlot.setup_html()

---
## Task 1: Build the Resource class

Create a `CoffeeShop` class following the modeling convention. It should:

- Store the SimPy environment
- Create a `simpy.Resource` with `num_baristas` capacity (attribute name: `barista`)
- Store the `prep_time` as an attribute
- Have a `prepare_drink(self, customer_name)` method that:
  - Yields a timeout of `self.prep_time` minutes
  - Prints: `"{customer_name}'s drink is ready!"`

In [2]:
class CoffeeShop:

    def __init__(self, env, num_baristas, prep_time):
        self.env = env
        self.barista = simpy.Resource(env, num_baristas)
        self.prep_time = prep_time
        
    def prepare_drink(self, customer_name):
        yield self.env.timeout(self.prep_time)
        print(f"{customer_name}'s drink is ready!")

---
## Task 2: Build the Entity class

Create a `Customer` class following the modeling convention. It should:

- Store the environment and the customer's name
- Have a `run(self, coffee_shop)` method that models the customer's flow:
  1. Print: `"{name} enters the coffee shop at {time:.2f}"`
  2. Request a barista (`coffee_shop.barista`)
  3. Once a barista is available, print: `"{name} places an order at {time:.2f}"`
  4. Yield the `prepare_drink` process
  5. Print: `"{name} leaves at {time:.2f}"`

**Hint:** Use the `with coffee_shop.barista.request() as request` pattern from the lecture.

In [3]:
class Customer:

    def __init__(self, env, customer_name):
        self.env = env
        self.name = customer_name

    def run(self, coffee_shop):
        print(f'{self.name} enters the coffee shop at {self.env.now:.2f}.')
        with coffee_shop.barista.request() as request:
            yield request #wait for prep to be finished

            print(f'{self.name} places an order at {self.env.now:.2f}.')
            yield self.env.process(coffee_shop.prepare_drink(self.name))

            print(f'{self.name} leaves at {self.env.now:.2f}.')

---
## Task 3: Build the Generator function

Create a `customer_generator(env, t_inter, coffee_shop)` function that:

1. Creates 5 initial customers ("Customer 0" through "Customer 4") and registers their `run` process
2. Then continuously generates new customers:
   - Wait a random time between `t_inter - 1` and `t_inter + 1` minutes
   - Create a new customer and register their `run` process

**Hint:** Use `random.randint(t_inter - 1, t_inter + 1)` for the inter-arrival time.

In [4]:
def customer_generator(env, t_inter, coffee_shop):
    customer_count = 0

    # Create 5 initial customers
    for _ in range(5):
        customer = Customer(env, customer_name=f'Customer {customer_count}')
        customer_count += 1
        env.process(customer.run(coffee_shop))

    # Create more customers while the simulation is running
    while True:
        yield env.timeout(random.randint(t_inter - 1, t_inter + 1))
        customer = Customer(env, customer_name=f'Customer {customer_count}')
        customer_count += 1
        env.process(customer.run(coffee_shop))

---
## Task 4: Run the simulation

Follow the **5-step pattern** from the lecture to run the simulation:

1. Define parameters
2. Create the SimPy environment
3. Create the resource (coffee shop)
4. Start the customer generator process
5. Execute the simulation

Use the parameter values from the scenario table above. Don't forget to set the random seed.

In [5]:
# 1. Define parameters
import random
RANDOM_SEED = 42
NUM_BARISTAS = 2
PREP_TIME = 5
T_INTER = 3
SIM_TIME = 60

# 2. Create environment
random.seed(RANDOM_SEED)
env = simpy.Environment()

# 3. Create resources
coffee_shop = CoffeeShop(env, NUM_BARISTAS, PREP_TIME)

# 4. Start the generator
env.process(customer_generator(env, T_INTER, coffee_shop))

# 5. Execute simulation
print('Running Simulation...')
random.seed(RANDOM_SEED)  # This helps to reproduce the results (more on this later)
env.run(until=SIM_TIME)
print('... Done')

Running Simulation...
Customer 0 enters the coffee shop at 0.00.
Customer 1 enters the coffee shop at 0.00.
Customer 2 enters the coffee shop at 0.00.
Customer 3 enters the coffee shop at 0.00.
Customer 4 enters the coffee shop at 0.00.
Customer 0 places an order at 0.00.
Customer 1 places an order at 0.00.
Customer 5 enters the coffee shop at 4.00.
Customer 0's drink is ready!
Customer 1's drink is ready!
Customer 0 leaves at 5.00.
Customer 1 leaves at 5.00.
Customer 2 places an order at 5.00.
Customer 3 places an order at 5.00.
Customer 6 enters the coffee shop at 6.00.
Customer 7 enters the coffee shop at 8.00.
Customer 2's drink is ready!
Customer 3's drink is ready!
Customer 2 leaves at 10.00.
Customer 3 leaves at 10.00.
Customer 4 places an order at 10.00.
Customer 5 places an order at 10.00.
Customer 8 enters the coffee shop at 12.00.
Customer 4's drink is ready!
Customer 5's drink is ready!
Customer 9 enters the coffee shop at 15.00.
Customer 4 leaves at 15.00.
Customer 5 leave

---
## Task 5: Variable preparation time

In reality, not every drink takes the same time. Modify the `prepare_drink` method so that the preparation time is **random**: between `prep_time - 2` and `prep_time + 2` minutes.

Redefine the full `CoffeeShop` class below, then re-run the simulation.

**Hint:** Use `random.randint()` to draw a random integer (type `random.randint?` in a code cell to see the documentation).

In [6]:
# Your code here
class CoffeeShop:

    def __init__(self, env, num_baristas, prep_time):
        self.env = env
        self.barista = simpy.Resource(env, num_baristas)
        self.prep_time = prep_time
        
    def prepare_drink(self, customer_name):
        new_prep_time = random.randint(self.prep_time - 2, self.prep_time + 2)
        yield self.env.timeout(new_prep_time)
        print(f"{customer_name}'s drink is ready!")

In [7]:
# Re-run with variable prep time
RANDOM_SEED = 42
NUM_BARISTAS = 2
PREP_TIME = 5
T_INTER = 3
SIM_TIME = 60
random.seed(RANDOM_SEED)
env = simpy.Environment()
coffee_shop = CoffeeShop(env, NUM_BARISTAS, PREP_TIME)
env.process(customer_generator(env, T_INTER, coffee_shop))

print('Running Simulation...')
env.run(until=SIM_TIME)
print('... Done')

Running Simulation...
Customer 0 enters the coffee shop at 0.00.
Customer 1 enters the coffee shop at 0.00.
Customer 2 enters the coffee shop at 0.00.
Customer 3 enters the coffee shop at 0.00.
Customer 4 enters the coffee shop at 0.00.
Customer 0 places an order at 0.00.
Customer 1 places an order at 0.00.
Customer 0's drink is ready!
Customer 1's drink is ready!
Customer 0 leaves at 3.00.
Customer 1 leaves at 3.00.
Customer 2 places an order at 3.00.
Customer 3 places an order at 3.00.
Customer 5 enters the coffee shop at 4.00.
Customer 6 enters the coffee shop at 6.00.
Customer 3's drink is ready!
Customer 3 leaves at 7.00.
Customer 4 places an order at 7.00.
Customer 2's drink is ready!
Customer 7 enters the coffee shop at 8.00.
Customer 2 leaves at 8.00.
Customer 5 places an order at 8.00.
Customer 4's drink is ready!
Customer 4 leaves at 10.00.
Customer 6 places an order at 10.00.
Customer 8 enters the coffee shop at 12.00.
Customer 6's drink is ready!
Customer 6 leaves at 13.00.

---
## Task 6: Add data collection with EventLogger

Printing is useful for debugging, but we need structured data for analysis.

**Step 1:** Copy the `EventLogger` class from the lecture.

**Step 2:** Modify the `Customer` class to:
- Accept a `logger` parameter in `__init__`
- At the end of the `run` method, call `self.logger.log(...)` with:
  - `customer`: the customer's name
  - `arrival_time`: when the customer entered
  - `queue_length`: how many customers were waiting when this customer arrived
  - `waiting_time`: how long the customer waited for a barista

**Step 3:** Update the `customer_generator` to accept and pass the `logger`.

**Step 4:** Run the simulation and display the resulting DataFrame.

In [8]:
class EventLogger:
    def __init__(self):
        self.logs = []

    def log(self, **kwargs):
        self.logs.append(kwargs)

    def get_df(self):
        return pd.DataFrame(self.logs)

In [9]:
# Your code here
class Customer:

    def __init__(self, env, customer_name, logger):
        self.env = env
        self.name = customer_name
        self.logger = logger

    def run(self, coffee_shop):
        print(f'{self.name} enters the coffee shop at {self.env.now:.2f}.')
        arrival_time = self.env.now
        queue_length = len(coffee_shop.barista.queue)
        with coffee_shop.barista.request() as request:
            yield request #wait for prep to be finished
            waiting_time = self.env.now - arrival_time
            
            
            yield self.env.process(coffee_shop.prepare_drink(self.name))

            self.logger.log(
                customer=self.name,
                arrival_time=arrival_time,
                queue_length=queue_length,
                waiting_time=waiting_time,
            )

In [10]:
# Your code here
def customer_generator(env, t_inter, coffee_shop, logger):
    customer_count = 0

    # Create 5 initial customers
    for _ in range(5):
        customer = Customer(env, customer_name=f'Customer {customer_count}', logger = logger)
        customer_count += 1
        env.process(customer.run(coffee_shop))

    # Create more customers while the simulation is running
    while True:
        yield env.timeout(random.randint(t_inter - 1, t_inter + 1))
        customer = Customer(env, customer_name=f'Customer {customer_count}', logger = logger)
        customer_count += 1
        env.process(customer.run(coffee_shop))

In [11]:
# Your code here
# Re-run with variable prep time
RANDOM_SEED = 2
NUM_BARISTAS = 2
PREP_TIME = 5
T_INTER = 3
SIM_TIME = 60
logger = EventLogger()
random.seed(RANDOM_SEED)
env = simpy.Environment()
coffee_shop = CoffeeShop(env, NUM_BARISTAS, PREP_TIME)
env.process(customer_generator(env, T_INTER, coffee_shop, logger))


env.run(until=SIM_TIME)

Customer 0 enters the coffee shop at 0.00.
Customer 1 enters the coffee shop at 0.00.
Customer 2 enters the coffee shop at 0.00.
Customer 3 enters the coffee shop at 0.00.
Customer 4 enters the coffee shop at 0.00.
Customer 5 enters the coffee shop at 2.00.
Customer 0's drink is ready!
Customer 1's drink is ready!
Customer 6 enters the coffee shop at 5.00.
Customer 2's drink is ready!
Customer 3's drink is ready!
Customer 7 enters the coffee shop at 8.00.
Customer 8 enters the coffee shop at 10.00.
Customer 9 enters the coffee shop at 12.00.
Customer 4's drink is ready!
Customer 5's drink is ready!
Customer 10 enters the coffee shop at 16.00.
Customer 6's drink is ready!
Customer 11 enters the coffee shop at 20.00.
Customer 7's drink is ready!
Customer 8's drink is ready!
Customer 12 enters the coffee shop at 24.00.
Customer 13 enters the coffee shop at 27.00.
Customer 9's drink is ready!
Customer 14 enters the coffee shop at 30.00.
Customer 10's drink is ready!
Customer 15 enters the 

---
## Task 7: Analyze the results

Using the DataFrame from Task 6, answer the following questions:

1. What is the **mean waiting time** across all customers? (Hint: use `.mean()`)
2. What is the **maximum queue length** any customer faced? (Hint: use `.max()`)
3. How many customers had a **waiting time of zero** (i.e., were served immediately)? (Hint: use `.sum()`)

### 7a) Bar chart: Waiting time per customer

Create a bar chart showing the waiting time for each customer. Fill in the column names for the x and y axes.

```python
(ggplot(df, aes(x=___, y=___))
 + geom_bar(stat='identity', fill='steelblue')
 + labs(x='Customer', y='Waiting time (min)', title='Waiting Time per Customer')
 + theme(axis_text_x=element_text(angle=45, hjust=1))
)
```

### 7b) Step plot: Queue length over time

Create a step plot showing how the queue length changed over time. Fill in the column names for the x and y axes, and choose the correct geom function.

```python
(ggplot(df, aes(x=___, y=___))
 + ___(color='tomato')
 + labs(x='Time (min)', y='Queue length at arrival', title='Queue Length Over Time')
)
```

In [12]:
# Your code here
df = logger.get_df()
from lets_plot import *
LetsPlot.setup_html()

mean_waiting_time = df['waiting_time'].mean()
max_que_length = df['queue_length'].max()
num_with_zero_wait = len(df[df['waiting_time'] == 0])
print(mean_waiting_time, max_que_length, num_with_zero_wait)

5.681818181818182 3 2


In [13]:
# Your code here
# Waiting time per customer
(ggplot(df, aes(x='customer', y='waiting_time'))
 + geom_bar(stat='identity', fill='steelblue')
 + labs(x='Customer', y='Waiting time (min)', title='Waiting Time per Customer')
 + theme(axis_text_x=element_text(angle=45, hjust=1))
)

In [14]:
# Your code here
(ggplot(df, aes(x='arrival_time', y='queue_length'))
 + geom_step(color='tomato')
 + labs(x='Time (min)', y='Queue length at arrival', title='Queue Length Over Time')
)

---
## Task 8: What if?

The coffee shop owner considers **hiring a third barista**. Run the simulation with `NUM_BARISTAS = 3` and compare the results:

1. How does the **mean waiting time** change?
2. How does the **maximum queue length** change?
3. Is hiring a third barista worth it based on the data?

In [15]:
# Your code here
RANDOM_SEED = 42
NUM_BARISTAS = 3
PREP_TIME = 5
T_INTER = 3
SIM_TIME = 60
logger = EventLogger()
random.seed(RANDOM_SEED)
env = simpy.Environment()
coffee_shop = CoffeeShop(env, NUM_BARISTAS, PREP_TIME)
env.process(customer_generator(env, T_INTER, coffee_shop, logger))


env.run(until=SIM_TIME)
df_new = logger.get_df()

Customer 0 enters the coffee shop at 0.00.
Customer 1 enters the coffee shop at 0.00.
Customer 2 enters the coffee shop at 0.00.
Customer 3 enters the coffee shop at 0.00.
Customer 4 enters the coffee shop at 0.00.
Customer 0's drink is ready!
Customer 1's drink is ready!
Customer 5 enters the coffee shop at 4.00.
Customer 2's drink is ready!
Customer 6 enters the coffee shop at 6.00.
Customer 3's drink is ready!
Customer 4's drink is ready!
Customer 5's drink is ready!
Customer 7 enters the coffee shop at 10.00.
Customer 8 enters the coffee shop at 12.00.
Customer 6's drink is ready!
Customer 9 enters the coffee shop at 15.00.
Customer 8's drink is ready!
Customer 7's drink is ready!
Customer 10 enters the coffee shop at 17.00.
Customer 9's drink is ready!
Customer 11 enters the coffee shop at 19.00.
Customer 10's drink is ready!
Customer 12 enters the coffee shop at 23.00.
Customer 13 enters the coffee shop at 25.00.
Customer 11's drink is ready!
Customer 14 enters the coffee shop at

---
## Task 9: Business perspective

The simulation shows that a third barista reduces waiting times. But does that mean hiring one is a good business decision?

**Discuss:** What factors determine whether hiring a third barista pays off for the coffee shop owner?

## Mentimeter

In [16]:
(ggplot(df_new, aes(x='customer', y='waiting_time'))
 + geom_bar(stat='identity', fill='steelblue')
 + labs(x='Customer', y='Waiting time (min)', title='Waiting Time per Customer')
 + theme(axis_text_x=element_text(angle=45, hjust=1))
)

In [17]:
# Your code here
(ggplot(df_new, aes(x='arrival_time', y='queue_length'))
 + geom_step(color='tomato')
 + labs(x='Time (min)', y='Queue length at arrival', title='Queue Length Over Time')
)

---

<img src="images/d3.png" style="width:50%; float:center;" />